# 🎬 Live Demo: Portfolio Backtesting System

**DADS 4002 - Database Systems Project**

**Interactive Analytics Demo with SQL Stored Procedures**

---

## 🎯 วิธีใช้งาน

1. **รัน Cell แรก** (Setup) - กด Shift+Enter
2. **รัน Cell ที่สอง** (Main Menu) - กด Shift+Enter **ครั้งเดียว**
3. **เลือก Option** 1-5 เพื่อ Demo Analytics Features
4. **เลือก 0** เพื่อออกจากโปรแกรม

---

## ⭐ Features

- ✅ **Interactive Menu** - เลือก Option ไปเรื่อยๆ ได้
- ✅ **SQL Stored Procedures** - ใช้ SQL เป็นหลัก (ตามโจทย์ข้อ 3)
- ✅ **Actionable Insights** - คำแนะนำการลงทุน (ตามโจทย์ข้อ 5f)
- ✅ **Live Demo** - เหมาะสำหรับนำเสนอ

---

## ⚠️ ก่อนเริ่ม

**ต้องรัน `setup_procedures.py` ก่อน** เพื่อติดตั้ง SQL Stored Procedures:

```bash
python3 setup_procedures.py
```

---

# 📦 Setup (Run Once)

**กด Shift+Enter เพื่อรัน Cell นี้**

In [ ]:
import mysql.connector
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

# MySQL Configuration
MYSQL_CONFIG = {
    'host': 'localhost',
    'user': 'root',
    'password': 'krittanut123456',
    'database': 'portfolio_backtesting'
}

# Connect to database
def get_db_connection():
    try:
        conn = mysql.connector.connect(**MYSQL_CONFIG)
        return conn
    except mysql.connector.Error as e:
        print(f"❌ Connection Error: {e}")
        return None

# Test connection
conn = get_db_connection()
if conn:
    cursor = conn.cursor(dictionary=True)
    
    # Get statistics
    cursor.execute("SELECT COUNT(*) as total FROM etf_master")
    etf_count = cursor.fetchone()['total']
    
    cursor.execute("SELECT COUNT(*) as total FROM benchmark_portfolios")
    portfolio_count = cursor.fetchone()['total']
    
    cursor.execute("SELECT COUNT(*) as total FROM price_history")
    price_count = cursor.fetchone()['total']
    
    print("="*80)
    print("🎬 Portfolio Backtesting System - Live Demo")
    print("="*80)
    print(f"✅ Connected to MySQL successfully!")
    print(f"📊 Database: {MYSQL_CONFIG['database']}")
    print(f"\n📈 Database Statistics:")
    print(f"   - ETFs: {etf_count:,}")
    print(f"   - Portfolios: {portfolio_count:,}")
    print(f"   - Price History Records: {price_count:,}")
    print("\n✅ Setup เสร็จสมบูรณ์! พร้อมรัน Cell ถัดไป")
    print("="*80)
    
    cursor.close()
    conn.close()
else:
    print("❌ ไม่สามารถเชื่อมต่อ Database ได้")
    print("💡 กรุณาตรวจสอบว่า MySQL Server กำลังทำงานอยู่")

---

# 🎯 Live Demo - Interactive Analytics Menu

## **กด Shift+Enter เพื่อเริ่ม Demo (ครั้งเดียว)**

**จากนั้น:**
- เลือก Option 1-5 เพื่อ Demo Analytics Features
- เลือก 0 เพื่อออกจากโปรแกรม

---

In [ ]:
def show_menu():
    """แสดง Analytics Menu"""
    print("\n" + "="*80)
    print("📈 Data Analytics Menu (SQL-based)")
    print("="*80)
    print("")
    print("1. 🏆 Top Performers - Top 5 ETFs ที่ดีที่สุด")
    print("2. ⚖️  Portfolio Comparison - เปรียบเทียบ Portfolios ทั้งหมด")
    print("3. 🔗 ETF Correlation - วิเคราะห์ความสัมพันธ์ระหว่าง ETFs")
    print("4. 📊 Sharpe Ratio Calculator - คำนวณ Sharpe Ratio")
    print("5. 📈 Top 10 ETFs Performance - ดูผลตอบแทนจาก SQL View")
    print("0. 🚪 Exit - ออกจากโปรแกรม")
    print("")
    print("="*80)

def top_performers(cursor):
    """Analytics #1: Top Performers using sp_get_top_performers"""
    print("\n🏆 Top 5 ETFs ที่มี Sharpe Ratio สูงสุด")
    print("="*80)
    print("📌 ใช้ SQL Stored Procedure: sp_get_top_performers")
    print("="*80)
    
    # Call Stored Procedure
    cursor.callproc('sp_get_top_performers', ['sharpe', 5, '2009-01-01', '2025-01-01'])
    
    # Fetch results
    for result in cursor.stored_results():
        rows = result.fetchall()
    
    if rows:
        print(f"\n{'Rank':<6} {'Ticker':<10} {'ETF Name':<30} {'Return':<12} {'Volatility':<12} {'Sharpe':<10}")
        print("-"*80)
        
        for i, row in enumerate(rows, 1):
            ticker = row['ticker_symbol']
            name = row['etf_name'][:28]
            ret = f"{row['annualized_return_pct']:.2f}%"
            vol = f"{row['annualized_volatility_pct']:.2f}%"
            sharpe = f"{row['sharpe_ratio']:.4f}"
            print(f"#{i:<5} {ticker:<10} {name:<30} {ret:<12} {vol:<12} {sharpe:<10}")
        
        # Actionable Insights
        top = rows[0]
        print("\n💡 Actionable Insights:")
        print(f"   - 🎯 {top['ticker_symbol']} มี Sharpe Ratio สูงสุด ({top['sharpe_ratio']:.4f})")
        print(f"   - 📊 ผลตอบแทนต่อปี: {top['annualized_return_pct']:.2f}%")
        print(f"   - 💼 แนะนำสำหรับนักลงทุนที่ต้องการผลตอบแทนดีเมื่อปรับความเสี่ยง")
        if top['sharpe_ratio'] > 1.0:
            print(f"   - ✅ Sharpe Ratio > 1.0 = ผลตอบแทนคุ้มค่ากับความเสี่ยง")
    else:
        print("❌ ไม่พบข้อมูล")

def portfolio_comparison(cursor):
    """Analytics #2: Portfolio Comparison using sp_compare_portfolios"""
    print("\n⚖️  เปรียบเทียบ Benchmark Portfolios")
    print("="*80)
    print("📌 ใช้ SQL Stored Procedure: sp_compare_portfolios")
    print("="*80)
    
    # Call Stored Procedure
    cursor.callproc('sp_compare_portfolios', ['2020-01-01', '2025-01-01'])
    
    # Fetch results
    for result in cursor.stored_results():
        rows = result.fetchall()
    
    if rows:
        print(f"\n{'Portfolio':<30} {'Risk Level':<15} {'Return':<12} {'Volatility':<12} {'Sharpe':<10}")
        print("-"*80)
        
        for row in rows:
            portfolio = row['benchmark_name'][:28]
            risk = row['risk_level'][:13]
            ret = f"{row['portfolio_return_pct']:.2f}%"
            vol = f"{row['annualized_volatility_pct']:.2f}%"
            sharpe = f"{row['sharpe_ratio']:.4f}"
            print(f"{portfolio:<30} {risk:<15} {ret:<12} {vol:<12} {sharpe:<10}")
        
        # Actionable Insights
        best = max(rows, key=lambda x: x['sharpe_ratio'])
        print("\n💡 Actionable Insights:")
        print(f"   - 🏆 Portfolio ที่ดีที่สุด: {best['benchmark_name']}")
        print(f"   - 📊 Sharpe Ratio: {best['sharpe_ratio']:.4f}")
        print(f"   - 💰 ผลตอบแทนต่อปี: {best['portfolio_return_pct']:.2f}%")
        print(f"   - 🎯 ระดับความเสี่ยง: {best['risk_level']}")
        print(f"   - 💼 แนะนำสำหรับนักลงทุนที่ต้องการ Risk-adjusted Return ที่ดี")
    else:
        print("❌ ไม่พบข้อมูล")

def etf_correlation(cursor):
    """Analytics #3: ETF Correlation using sp_get_etf_correlation"""
    print("\n🔗 ETF Correlation Analysis")
    print("="*80)
    print("📌 ใช้ SQL Stored Procedure: sp_get_etf_correlation")
    print("="*80)
    
    ticker1 = input("\nป้อน Ticker 1 (เช่น SPY): ").strip().upper()
    ticker2 = input("ป้อน Ticker 2 (เช่น QQQ): ").strip().upper()
    
    # Call Stored Procedure
    result_args = cursor.callproc('sp_get_etf_correlation', [ticker1, ticker2, 0])
    correlation = result_args[2]
    
    if correlation is not None:
        print(f"\n📊 {ticker1} vs {ticker2}")
        print("-"*80)
        print(f"Correlation: {correlation:.4f}")
        
        # Interpretation
        if correlation > 0.8:
            strength = "แข็งแกร่งมาก (Highly Correlated)"
            recommendation = "⚠️  ไม่แนะนำให้ถือทั้ง 2 ETFs ในพอร์ตเดียวกัน"
            reason = "มีความเสี่ยงคล้ายกันมาก ไม่ได้ช่วย Diversify"
        elif correlation > 0.5:
            strength = "แข็งแกร่งปานกลาง (Moderately Correlated)"
            recommendation = "⚠️  ควรพิจารณาสัดส่วนการถืออย่างรอบคอบ"
            reason = "มีความสัมพันธ์ปานกลาง อาจช่วย Diversify ได้บ้าง"
        elif correlation > 0:
            strength = "อ่อน (Weakly Correlated)"
            recommendation = "✅ เหมาะสำหรับถือร่วมกัน"
            reason = "ช่วย Diversify ความเสี่ยงได้ดี"
        else:
            strength = "ติดลบ (Negative Correlation)"
            recommendation = "✅ ดีมากสำหรับ Diversification"
            reason = "เคลื่อนไหวในทิศทางตรงข้าม ช่วยลดความเสี่ยง"
        
        print(f"\nความสัมพันธ์: {strength}")
        print("\n💡 Actionable Insights:")
        print(f"   - {recommendation}")
        print(f"   - เหตุผล: {reason}")
    else:
        print("\n❌ ไม่พบข้อมูลหรือ Ticker ไม่ถูกต้อง")
        print("💡 ตัวอย่าง Ticker: SPY, QQQ, AGG, GLD, VTI, BND")

def sharpe_ratio_calculator(cursor):
    """Analytics #4: Sharpe Ratio Calculator using sp_calculate_sharpe_ratio"""
    print("\n📊 Sharpe Ratio Calculator")
    print("="*80)
    print("📌 ใช้ SQL Stored Procedure: sp_calculate_sharpe_ratio")
    print("="*80)
    
    ticker = input("\nป้อน Ticker Symbol (เช่น SPY): ").strip().upper()
    
    try:
        risk_free = float(input("Risk-Free Rate (เช่น 0.02 = 2%): ").strip())
    except:
        risk_free = 0.02
        print(f"ใช้ค่า Default: {risk_free}")
    
    # Call Stored Procedure
    result_args = cursor.callproc('sp_calculate_sharpe_ratio', [ticker, risk_free, 0])
    sharpe = result_args[2]
    
    if sharpe is not None:
        print(f"\n📊 Sharpe Ratio Analysis: {ticker}")
        print("-"*80)
        print(f"Risk-Free Rate: {risk_free*100:.2f}%")
        print(f"Sharpe Ratio: {sharpe:.4f}")
        
        # Rating
        if sharpe > 2:
            rating = "Excellent (ยอดเยี่ยม)"
        elif sharpe > 1:
            rating = "Good (ดี)"
        elif sharpe > 0.5:
            rating = "Fair (พอใช้)"
        else:
            rating = "Poor (ควรหลีกเลี่ยง)"
        
        print(f"\n💡 Actionable Insights:")
        print(f"   - 🎯 Rating: {rating}")
        if sharpe > 1:
            print(f"   - ✅ Sharpe Ratio > 1.0 = ผลตอบแทนคุ้มค่ากับความเสี่ยง")
            print(f"   - 💼 เหมาะสำหรับการลงทุน")
        else:
            print(f"   - ⚠️  Sharpe Ratio < 1.0 = ควรพิจารณาความเสี่ยงอย่างรอบคอบ")
    else:
        print("\n❌ ไม่พบข้อมูลหรือ Ticker ไม่ถูกต้อง")
        print("💡 ตัวอย่าง Ticker: SPY, QQQ, AGG, GLD, VTI, BND")

def etf_performance_view(cursor):
    """Analytics #5: ETF Performance using SQL View"""
    print("\n📈 Top 10 ETFs Performance (from SQL View)")
    print("="*80)
    print("📌 ใช้ SQL View: vw_etf_performance")
    print("="*80)
    
    query = """
    SELECT 
        ticker_symbol,
        annualized_return_pct,
        annualized_volatility_pct,
        sharpe_ratio_approx
    FROM vw_etf_performance
    ORDER BY sharpe_ratio_approx DESC
    LIMIT 10
    """
    
    cursor.execute(query)
    rows = cursor.fetchall()
    
    if rows:
        print(f"\n{'Rank':<6} {'Ticker':<10} {'Return':<15} {'Volatility':<15} {'Sharpe Ratio':<15}")
        print("-"*80)
        
        for i, row in enumerate(rows, 1):
            ticker = row['ticker_symbol']
            ret = f"{row['annualized_return_pct']:.2f}%" if row['annualized_return_pct'] else 'N/A'
            vol = f"{row['annualized_volatility_pct']:.2f}%" if row['annualized_volatility_pct'] else 'N/A'
            sharpe = f"{row['sharpe_ratio_approx']:.4f}" if row['sharpe_ratio_approx'] else 'N/A'
            print(f"#{i:<5} {ticker:<10} {ret:<15} {vol:<15} {sharpe:<15}")
        
        print("\n💡 Actionable Insights:")
        print("   - ✅ ข้อมูลนี้คำนวณโดยใช้ SQL Window Functions (LAG, OVER)")
        print("   - ✅ ไม่ใช้ Python/pandas ในการคำนวณ")
        print("   - ✅ ตรงตามโจทย์อาจารย์ข้อ 3: ใช้ SQL เป็นเครื่องมือหลักในการวิเคราะห์")
    else:
        print("❌ ไม่พบข้อมูล")

# Main Interactive Loop
def main_loop():
    """Main interactive menu loop"""
    print("\n" + "="*80)
    print("🎬 Live Demo Started!")
    print("="*80)
    print("💡 กรุณาเลือก Option 1-5 เพื่อ Demo Analytics Features")
    print("💡 เลือก 0 เพื่อออกจากโปรแกรม")
    
    conn = get_db_connection()
    if not conn:
        print("❌ ไม่สามารถเชื่อมต่อ Database ได้")
        return
    
    cursor = conn.cursor(dictionary=True)
    
    while True:
        try:
            show_menu()
            choice = input("\nเลือก Option (0-5): ").strip()
            
            if choice == '0':
                print("\n" + "="*80)
                print("🚪 ขอบคุณที่ใช้งาน Portfolio Backtesting System!")
                print("="*80)
                print("\n✅ สรุปการ Demo:")
                print("   - ใช้ SQL Stored Procedures ทั้งหมด (ตามโจทย์ข้อ 3)")
                print("   - มี Actionable Insights ในทุก Feature (ตามโจทย์ข้อ 5f)")
                print("   - ข้อมูลจริงจาก Yahoo Finance (ตามโจทย์ข้อ 4)")
                print("\n💼 ระบบหลักอยู่ที่: main.py (Integrated System)")
                print("\n👋 Goodbye!\n")
                break
            
            elif choice == '1':
                top_performers(cursor)
            
            elif choice == '2':
                portfolio_comparison(cursor)
            
            elif choice == '3':
                etf_correlation(cursor)
            
            elif choice == '4':
                sharpe_ratio_calculator(cursor)
            
            elif choice == '5':
                etf_performance_view(cursor)
            
            else:
                print("\n⚠️  กรุณาเลือก Option 0-5 เท่านั้น")
            
            input("\n⏎ กด Enter เพื่อกลับสู่ Menu...")
            
        except KeyboardInterrupt:
            print("\n\n⚠️  Interrupted by user")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")
            input("\n⏎ กด Enter เพื่อกลับสู่ Menu...")
    
    # Close connection
    cursor.close()
    conn.close()
    print("\n✅ Database connection closed")

# Start the interactive loop
main_loop()

---

# ✅ Demo เสร็จสิ้น

## 📊 สรุป

Notebook นี้แสดงให้เห็นว่า:

### **ข้อ 3: ใช้ SQL เป็นเครื่องมือหลักในการวิเคราะห์** ✅
- ใช้ **5 SQL Stored Procedures**: 
  - `sp_get_top_performers`
  - `sp_compare_portfolios`
  - `sp_get_etf_correlation`
  - `sp_calculate_sharpe_ratio`
  
- ใช้ **SQL View**: `vw_etf_performance`

### **ข้อ 5f: Actionable Insights** ✅
- ทุก Analytics Feature มี **คำแนะนำการลงทุน**
- บอกว่าควรทำอย่างไร ไม่ได้แค่แสดงตัวเลข

### **Interactive Live Demo** ✅
- รัน Cell เดียว แล้ววนลูปได้เรื่อยๆ
- เหมาะสำหรับนำเสนอต่ออาจารย์

---

## 📝 หมายเหตุ

- **Notebook นี้เป็น Live Demo** สำหรับนำเสนอเท่านั้น
- **ระบบหลักอยู่ที่ `main.py`** - Integrated System ตามโจทย์อาจารย์ข้อ 1 และ 5a

---

**Thank you! 🎉**